# Five-model permutation feature importance — LiftWorks Senior Design II

Reproduces the **method** reported in the final project manuscript: five fixed-settings regression models (RF, KNN, DT, MLP, SVR); 12 base airfoils for training and the four independent test geometries for interpretation; **one shared 500-observation held-out subset**; **20 shuffles per feature**; permutation score = decrease in the mean of the three separate coefficient R² scores; normalize each model by its largest feature importance (maximum = **100 relative units**, **not percentages**).

This notebook does **not** retune hyperparameters. It refits the finalized models to recreate the permutation calculations; previously published numeric scores should only be treated as exactly reproducible once the same sampled rows, random seed, and library versions are verified. Feature importance is supplementary *post-test interpretation*, not a model-selection procedure.

The previously uploaded RF-only `features_importance-2` notebook is an **older, different** impurity-importance analysis and should not be identified as the source of the publication's five-model permutation figure.

In [ ]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
from sklearn.multioutput import MultiOutputRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import sklearn

SEED = 42
N_HELD_OUT = 500
N_PERMUTATIONS = 20
FEATURES = ['%Camber', '%Thickness', 'Reynolds#', 'Mach#', 'Alpha']
TARGETS = ['cl', 'cd', 'cm']
MODEL_NAMES = ['RF', 'KNN', 'DT', 'MLP', 'SVR']

# Select the local dataset or the original Colab Drive location.
DATA_CANDIDATES = [
    Path('data/PALMO_NACA_4Series_Complete_Dataset.xlsx'),
    Path('PALMO_NACA_4Series_Complete_Dataset.xlsx'),
    Path('/content/drive/MyDrive/Test_data/PALMO_NACA_4Series_Complete_Dataset.xlsx'),
]
DATA_FILE = next((p for p in DATA_CANDIDATES if p.exists()), None)
if DATA_FILE is None:
    raise FileNotFoundError('Place the dataset in data/ or set DATA_FILE to its location.')
print('Dataset:', DATA_FILE)
print('scikit-learn version:', sklearn.__version__)


In [ ]:
df = pd.read_excel(DATA_FILE, sheet_name='PALMO_4Series_Data')
X = df[FEATURES].copy()
Y = df[TARGETS].copy()

# Four airfoils never used for training or hyperparameter selection.
test_mask = (
    ((df['%Camber'] == 3) & (df['%Thickness'].isin([15, 18]))) |
    ((df['%Camber'] == 4) & (df['%Thickness'].isin([15, 21])))
)
X_base, Y_base = X.loc[~test_mask], Y.loc[~test_mask]
X_test, Y_test = X.loc[test_mask], Y.loc[test_mask]
assert len(X_base) == 39360 and len(X_test) == 13120
assert len(X_test) >= N_HELD_OUT

# Deterministic, shared random sample for all five models.
rng = np.random.default_rng(SEED)
sample_positions = np.sort(rng.choice(len(X_test), size=N_HELD_OUT, replace=False))
X_eval = X_test.iloc[sample_positions].copy()
Y_eval = Y_test.iloc[sample_positions].copy()
print('Development rows:', len(X_base), '| Held-out rows:', len(X_test))
print('Shared permutation-evaluation sample:', len(X_eval))


In [ ]:
def create_model(method, random_state):
    if method == 'RF':
        return RandomForestRegressor(
            n_estimators=300,
            max_depth=20,
            min_samples_leaf=1,
            min_samples_split=2,
            max_features=1.0,
            random_state=random_state,
            n_jobs=-1
        )

    if method == 'DT':
        return DecisionTreeRegressor(
            max_depth=20,
            min_samples_leaf=1,
            min_samples_split=2,
            random_state=random_state
        )

    if method == 'KNN':
        return KNeighborsRegressor(
            n_neighbors=3,
            weights='distance',
            p=1,
            n_jobs=-1
        )

    if method == 'SVR':
        return TransformedTargetRegressor(
            regressor=MultiOutputRegressor(
                SVR(
                    kernel='rbf',
                    C=10.0,
                    epsilon=0.05,
                    gamma=0.2,
                    cache_size=256,
                    tol=0.001,
                    max_iter=-1
                ),
                n_jobs=1
            ),
            transformer=StandardScaler()
        )

    if method == 'MLP':
        return MLPRegressor(
            hidden_layer_sizes=(128, 64),
            alpha=0.0001,
            learning_rate_init=0.003,
            max_iter=500,
            random_state=random_state
        )

    raise ValueError('Unknown model: ' + str(method))


In [ ]:
def mean_output_r2(y_true, y_pred):
    """Mean of the individual Cl, Cd and Cm R² scores."""
    return float(np.mean([r2_score(y_true[:, j], y_pred[:, j]) for j in range(3)]))

all_rows = []
for model_name in MODEL_NAMES:
    print('Fitting', model_name, '...', flush=True)
    model = create_model(model_name, random_state=0)
    if model_name in ['KNN', 'MLP', 'SVR']:
        scaler = StandardScaler().fit(X_base)
        X_fit = scaler.transform(X_base)
        X_eval_fit = scaler.transform(X_eval)
    else:
        scaler = None
        X_fit = X_base
        X_eval_fit = X_eval

    t0 = time.perf_counter()
    model.fit(X_fit, Y_base.to_numpy())
    print('Fit time (s):', round(time.perf_counter()-t0, 2))
    truth = Y_eval.to_numpy()
    baseline = mean_output_r2(truth, model.predict(X_eval_fit))

    # Start with the same scaled/unscaled values the model receives.
    evaluation_array = np.asarray(X_eval_fit).copy()
    for j, feature in enumerate(FEATURES):
        drops = []
        for k in range(N_PERMUTATIONS):
            # Deterministic independent shuffle for this model/feature/repetition.
            shuffled = evaluation_array.copy()
            shuffled[:, j] = rng.permutation(shuffled[:, j])
            permuted_r2 = mean_output_r2(truth, model.predict(shuffled))
            drops.append(baseline - permuted_r2)
        all_rows.append({'Model':model_name, 'Feature':feature,
                         'Mean_R2_decrease':float(np.mean(drops)),
                         'Std_R2_decrease':float(np.std(drops)),
                         'Baseline_mean_R2':baseline})

raw = pd.DataFrame(all_rows)
# Relative max=100 within each model. This is not a share that sums to 100%.
max_by_model = raw.groupby('Model')['Mean_R2_decrease'].transform('max')
raw['Relative_importance_max100'] = np.where(max_by_model > 0, 100 * raw['Mean_R2_decrease'] / max_by_model, np.nan)
summary = raw.pivot(index='Feature', columns='Model', values='Relative_importance_max100').loc[FEATURES, MODEL_NAMES]
display(summary.round(2))


In [ ]:
outdir = Path('results')
figdir = Path('figures')
outdir.mkdir(exist_ok=True)
figdir.mkdir(exist_ok=True)
raw.to_csv(outdir/'five_model_permutation_importance_raw.csv', index=False)
summary.to_csv(outdir/'five_model_permutation_importance_normalized.csv')

plot_data = summary.loc[['Alpha','Mach#','%Thickness','%Camber','Reynolds#']]
ax = plot_data.plot(kind='bar', figsize=(11,6), width=0.82)
ax.set_ylabel('Relative permutation importance (within-model max = 100)')
ax.set_xlabel('Input feature')
ax.set_title('Permutation Importance Across Five Regression Models')
ax.set_ylim(bottom=0)
ax.legend(title='Model', ncol=5)
ax.tick_params(axis='x', labelrotation=0)
ax.grid(axis='y', alpha=0.2)
plt.tight_layout()
plt.savefig(figdir/'five_model_permutation_importance.png', dpi=300, bbox_inches='tight')
plt.savefig(figdir/'five_model_permutation_importance.svg', bbox_inches='tight')
plt.show()
print('Saved CSV results to', outdir, 'and publication figure to', figdir)
